In [0]:
df_csv=spark.read.format("csv")\
            .option("header",True)\
            .option("inferSchema",True)\
            .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

display(df_csv)

#Delta format

In [0]:
df_csv.write.format("delta")\
            .mode("append")\
            .option("path","/Volumes/sparkcatalog/raw/destination/files/delta")\
            .save()

#Upsert with Delta Library

In [0]:
display(df_csv)

In [0]:
from pyspark.sql.functions import col,lit

df_new_0=df_csv.filter(col("order_id").isin(1001,1002))

df_new_1=df_new_0.filter(col("order_id")==1001)\
              .withColumn("product_id",lit("P1012"))

df_new_2=df_new_0.filter(col("order_id")==1002)\
                .withColumn("order_id",lit("9000"))


df_new=df_new_1.union(df_new_2)

display(df_new)

In [0]:
from delta.tables import DeltaTable

dlt_obj=DeltaTable.forPath(spark,"/Volumes/sparkcatalog/raw/destination/files/delta/")

dlt_obj.alias("tgt").merge(df_new.alias("src")\
                    ,"tgt.order_id=src.order_id")\
                    .whenMatchedUpdateAll()\
                    .whenNotMatchedInsertAll()\
                    .execute()

In [0]:
df_read_delta=spark.read.format("delta")\
                    .option("header",True)\
                    .option("inferSchema",True)\
                    .load("/Volumes/sparkcatalog/raw/destination/files/delta/")
display(df_read_delta)

In [0]:
%sql
create volume if not exists sparkcatalog.raw.json_files;

#Json Files

In [0]:
path="/Volumes/sparkcatalog/raw/json_files/day1.json"

In [0]:
from pyspark.sql.functions import explode,col

df_json=spark.read.format("json")\
                 .option("inferSchema",True)\
                 .option("multiLine",True)\
                 .load(path)


df_json=df_json.withColumn("items",explode(col("items")))

df_json=df_json.withColumn("item_id",col("items.item_id"))\
                .withColumn("item_price",col("items.price"))\
                .withColumn("Product_name",col("items.product_name"))\
                .withColumn("quantity",col("items.quantity"))\
                .drop("items")\
                .withColumn("customer_address",col("customer.customer_id"))\
                .withColumn("email",col("customer.email"))\
                .withColumn("name",col("customer.name"))\
                .withColumn("customer_address_city",col("customer.address.city"))\
                .withColumn("customer_address_country",col("customer.address.country"))\
                .withColumn("customer_address_postal_code",col("customer.address.postal_code"))\
                .drop("customer")\
                .withColumn("payment_method",col("payment.method"))\
                .withColumn("payment_trasaction_id",col("payment.transaction_id"))\
                .drop("payment")\
                .drop("metadata")

display(df_json)
                 